# PyQuil Error Mitigation Workflow

This executed notebook completes a compact quantum error-mitigation workflow.

| Step | Work completed here |
|---|---|
| Step 1 | Build a Bell-state circuit, simulate ideal counts, compute $\langle ZZ \rangle$ from counts |
| Step 2 | Add simple readout noise, compare ideal vs noisy counts, record raw noisy $\langle ZZ \rangle$ |
| Step 3 | Build a 2-qubit readout confusion matrix, apply readout mitigation, compare raw vs mitigated $\langle ZZ \rangle$ |
| Step 4 | Implement CNOT folding with 1, 3, and 5 CNOTs, run a simple noisy simulator, record $\langle ZZ \rangle$ at each scale |
| Step 5 | Fit a linear ZNE curve, estimate zero-noise $\langle ZZ \rangle$, and make a final comparison table |

The code is written in PyQuil and kept intentionally simple. NumPy is used for sampling, matrices, and the linear fit.

## Cell 1: Imports and Constants

This cell imports PyQuil and NumPy, then sets the global constants used across the notebook.

We use the bitstring order `00`, `01`, `10`, `11` everywhere. For $ZZ$, outcomes `00` and `11` contribute `+1`, while `01` and `10` contribute `-1`.

In [1]:
import numpy as np

from pyquil import Program
from pyquil.gates import CNOT, H
from pyquil.simulation.tools import program_unitary


BITSTRINGS = ["00", "01", "10", "11"]
SHOTS = 8192
SEED = 7

READOUT_ERROR = 0.08
CNOT_GATE_ERROR = 0.02

np.set_printoptions(precision=4, suppress=True)

## Cell 2: Helper Functions

This cell defines small helper functions used later:

- `bell_program` builds a PyQuil Bell circuit with a chosen number of CNOTs.
- `state_probabilities` gets ideal probabilities from a PyQuil program.
- `sample_counts` produces reproducible counts from probabilities.
- `zz_from_counts` and `zz_from_probabilities` compute $\langle ZZ \rangle$.
- `confusion_matrix` builds the 2-qubit readout-error matrix.
- `mitigate_counts` applies basic readout mitigation.
- `simple_gate_noise` is a lightweight gate-noise model for the CNOT-folding/ZNE section.

In [2]:
def bell_program(cnot_count=1):
    program = Program(H(0))
    for _ in range(cnot_count):
        program += CNOT(0, 1)
    return program


def state_probabilities(program):
    unitary = program_unitary(program, n_qubits=2)
    initial_state = np.array([1, 0, 0, 0], dtype=complex)
    state = unitary @ initial_state
    probabilities = np.abs(state) ** 2
    return {bitstring: float(prob) for bitstring, prob in zip(BITSTRINGS, probabilities)}


def sample_counts(probabilities, shots=SHOTS, seed=SEED):
    rng = np.random.default_rng(seed)
    probs = np.array([probabilities.get(bitstring, 0.0) for bitstring in BITSTRINGS])
    samples = rng.multinomial(shots, probs / probs.sum())
    return dict(zip(BITSTRINGS, samples.tolist()))


def zz_from_counts(counts):
    shots = sum(counts.values())
    same = counts.get("00", 0) + counts.get("11", 0)
    different = counts.get("01", 0) + counts.get("10", 0)
    return (same - different) / shots


def zz_from_probabilities(probabilities):
    same = probabilities.get("00", 0.0) + probabilities.get("11", 0.0)
    different = probabilities.get("01", 0.0) + probabilities.get("10", 0.0)
    return same - different


def readout_probability(observed, true, error):
    probability = 1.0
    for observed_bit, true_bit in zip(observed, true):
        probability *= (1 - error) if observed_bit == true_bit else error
    return probability


def confusion_matrix(error):
    return np.array([
        [readout_probability(observed, true, error) for true in BITSTRINGS]
        for observed in BITSTRINGS
    ])


def apply_bit_flip_noise(probabilities, error):
    vector = np.array([probabilities.get(bitstring, 0.0) for bitstring in BITSTRINGS])
    noisy_vector = confusion_matrix(error) @ vector
    return {bitstring: float(prob) for bitstring, prob in zip(BITSTRINGS, noisy_vector)}


def mitigate_counts(counts, error=READOUT_ERROR):
    shots = sum(counts.values())
    observed = np.array([counts.get(bitstring, 0) / shots for bitstring in BITSTRINGS])
    mitigated = np.linalg.solve(confusion_matrix(error), observed)

    mitigated = np.clip(mitigated, 0.0, 1.0)
    mitigated = mitigated / mitigated.sum()

    return {bitstring: float(prob) for bitstring, prob in zip(BITSTRINGS, mitigated)}


def simple_gate_noise(probabilities, cnot_count, error=CNOT_GATE_ERROR):
    noisy_probabilities = probabilities
    for _ in range(cnot_count):
        noisy_probabilities = apply_bit_flip_noise(noisy_probabilities, error)
    return noisy_probabilities


def print_table(rows, headers):
    widths = [len(header) for header in headers]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(str(value)))

    line = " | ".join(header.ljust(widths[index]) for index, header in enumerate(headers))
    rule = "-+-".join("-" * width for width in widths)
    print(line)
    print(rule)
    for row in rows:
        print(" | ".join(str(value).ljust(widths[index]) for index, value in enumerate(row)))

## Cell 3: Bell Circuit, Ideal Counts, and $\langle ZZ \rangle$

This cell builds the Bell-state circuit in PyQuil, computes the ideal probabilities, samples ideal counts, and computes $\langle ZZ \rangle$ manually from those counts.

In [3]:
bell = bell_program(cnot_count=1)
ideal_probabilities = state_probabilities(bell)
ideal_counts = sample_counts(ideal_probabilities, seed=SEED)
ideal_zz = zz_from_counts(ideal_counts)

print("Bell circuit:")
print(bell)
print("Ideal probabilities:", ideal_probabilities)
print("Ideal counts:", ideal_counts)
print("Ideal <ZZ> from counts:", ideal_zz)

Bell circuit:
H 0
CNOT 0 1

Ideal probabilities: {'00': 0.4999999999999999, '01': 0.0, '10': 0.0, '11': 0.4999999999999999}
Ideal counts: {'00': 4090, '01': 0, '10': 0, '11': 4102}
Ideal <ZZ> from counts: 1.0


## Cell 4: Add Simple Readout Noise

This cell applies a simple independent readout-error model. Each measured bit has an 8% chance of being flipped. Then we compare ideal counts with noisy counts and record the raw noisy $\langle ZZ \rangle$.

In [4]:
readout_noisy_probabilities = apply_bit_flip_noise(ideal_probabilities, READOUT_ERROR)
readout_noisy_counts = sample_counts(readout_noisy_probabilities, seed=SEED)
raw_readout_noisy_zz = zz_from_counts(readout_noisy_counts)

print("Ideal counts:", ideal_counts)
print("Readout-noisy probabilities:", readout_noisy_probabilities)
print("Readout-noisy counts:", readout_noisy_counts)
print("Raw readout-noisy <ZZ>:", raw_readout_noisy_zz)

Ideal counts: {'00': 4090, '01': 0, '10': 0, '11': 4102}
Readout-noisy probabilities: {'00': 0.4263999999999999, '01': 0.07359999999999998, '10': 0.07359999999999998, '11': 0.4263999999999999}
Readout-noisy counts: {'00': 3487, '01': 584, '10': 585, '11': 3536}
Raw readout-noisy <ZZ>: 0.714599609375


## Cell 5: Confusion Matrix and Readout Mitigation

This cell builds the 2-qubit readout confusion matrix and uses it to estimate the probabilities before readout noise. Then it compares raw noisy $\langle ZZ \rangle$ with readout-mitigated $\langle ZZ \rangle$.

In [5]:
readout_matrix = confusion_matrix(READOUT_ERROR)
mitigated_probabilities = mitigate_counts(readout_noisy_counts)
readout_mitigated_zz = zz_from_probabilities(mitigated_probabilities)

print("Rows: observed 00, 01, 10, 11")
print("Cols: true     00, 01, 10, 11")
print(readout_matrix)
print("Mitigated probabilities:", mitigated_probabilities)
print("Raw readout-noisy <ZZ>:", raw_readout_noisy_zz)
print("Readout-mitigated <ZZ>:", readout_mitigated_zz)

Rows: observed 00, 01, 10, 11
Cols: true     00, 01, 10, 11
[[0.8464 0.0736 0.0736 0.0064]
 [0.0736 0.8464 0.0064 0.0736]
 [0.0736 0.0064 0.8464 0.0736]
 [0.0064 0.0736 0.0736 0.8464]]
Mitigated probabilities: {'00': 0.49646217754931565, '01': 0.0, '10': 0.0, '11': 0.5035378224506842}
Raw readout-noisy <ZZ>: 0.714599609375
Readout-mitigated <ZZ>: 0.9999999999999999


## Cell 6: CNOT Folding at Noise Scales 1, 3, and 5

This cell implements CNOT folding. Because CNOT is its own inverse, odd numbers of CNOTs preserve the ideal logical circuit:

$$\text{CNOT}^1 = \text{CNOT}, \qquad \text{CNOT}^3 = \text{CNOT}, \qquad \text{CNOT}^5 = \text{CNOT}.$$

The ideal operation stays the same, but the noisy operation accumulates more gate noise. This gives the noise-scale data needed for ZNE.

In [6]:
scale_factors = np.array([1, 3, 5])
zne_rows = []
zne_values = []

for scale in scale_factors:
    folded_program = bell_program(cnot_count=int(scale))
    folded_ideal_probabilities = state_probabilities(folded_program)
    folded_noisy_probabilities = simple_gate_noise(folded_ideal_probabilities, cnot_count=int(scale))
    folded_noisy_counts = sample_counts(folded_noisy_probabilities, seed=SEED + int(scale))
    folded_zz = zz_from_counts(folded_noisy_counts)

    zne_values.append(float(folded_zz))
    zne_rows.append([int(scale), folded_noisy_counts, round(float(folded_zz), 6)])

print_table(zne_rows, headers=["CNOT count / scale", "Noisy counts", "Noisy <ZZ>"])

CNOT count / scale | Noisy counts                                   | Noisy <ZZ>
-------------------+------------------------------------------------+-----------
1                  | {'00': 3882, '01': 156, '10': 179, '11': 3975} | 0.918213  
3                  | {'00': 3522, '01': 461, '10': 481, '11': 3728} | 0.77002   
5                  | {'00': 3351, '01': 698, '10': 707, '11': 3436} | 0.656982  


## Cell 7: Linear ZNE Fit

This cell fits a straight line through the folded-circuit data at scale factors 1, 3, and 5.

If the fitted line is

$$f(s) = ms + b,$$

then the zero-noise estimate is the intercept:

$$f(0) = b.$$

In [7]:
zne_values = np.array(zne_values)
slope, intercept = np.polyfit(scale_factors, zne_values, deg=1)
zne_estimate = float(intercept)

print("Scale factors:", scale_factors.tolist())
print("Measured noisy <ZZ> values:", [round(float(value), 6) for value in zne_values])
print("Linear fit: <ZZ>(scale) =", round(float(slope), 6), "* scale +", round(float(intercept), 6))
print("Zero-noise estimate from ZNE:", round(zne_estimate, 6))

Scale factors: [1, 3, 5]
Measured noisy <ZZ> values: [0.918213, 0.77002, 0.656982]
Linear fit: <ZZ>(scale) = -0.065308 * scale + 0.977661
Zero-noise estimate from ZNE: 0.977661


## Cell 8: Final Comparison Table

This cell creates the final comparison table: ideal, noisy, readout-mitigated, and ZNE estimates of $\langle ZZ \rangle$.

In [8]:
comparison_rows = [
    ["Ideal Bell state", round(ideal_zz, 6), "No noise; counts sampled from ideal probabilities"],
    ["Raw readout-noisy", round(raw_readout_noisy_zz, 6), "Readout bit-flip noise only"],
    ["Readout-mitigated", round(readout_mitigated_zz, 6), "Confusion-matrix correction applied"],
    ["ZNE estimate", round(zne_estimate, 6), "Linear extrapolation using CNOT scales 1, 3, 5"],
]

print_table(comparison_rows, headers=["Case", "<ZZ>", "Meaning"])

Case              | <ZZ>     | Meaning                                          
------------------+----------+--------------------------------------------------
Ideal Bell state  | 1.0      | No noise; counts sampled from ideal probabilities
Raw readout-noisy | 0.7146   | Readout bit-flip noise only                      
Readout-mitigated | 1.0      | Confusion-matrix correction applied              
ZNE estimate      | 0.977661 | Linear extrapolation using CNOT scales 1, 3, 5   


## Completed Work

The notebook completes the full workflow:

- Bell-state simulation and manual $\langle ZZ \rangle$ from counts
- simple readout-noise simulation
- 2-qubit readout confusion matrix
- basic readout mitigation
- CNOT folding with scale factors 1, 3, and 5
- linear zero-noise extrapolation
- final comparison table